In [ ]:
import os, sys
# Add project root to sys.path so we can import models and pipelines
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)


# PRISM — CNN Cough Detector Fine-Tuning

Fine-tunes the pre-trained ResNet-18 cough detector with **waveform-domain augmentation** to make it robust to browser-microphone recordings.

### What this notebook does:
1. Loads the existing `cough_detector_best.pt` checkpoint
2. Applies `MicAugment` transforms (reverb, noise, gain, codec, bandpass) to simulate mic conditions
3. Fine-tunes with frozen early layers (first 5 epochs) then full fine-tuning
4. Saves `cough_detector_finetuned.pt` — the original checkpoint is NEVER modified

### Prerequisites:
- Upload `datasets-features.zip` to Google Drive
- Upload `prism-colab/` folder to Google Drive
- Upload `models/checkpoints/cough_detector_best.pt` to Google Drive

**Runtime → Change runtime type → T4 GPU**

## 1. Setup & Mount Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Set your Google Drive path (adjust if needed)
DRIVE_ROOT = '/content/drive/MyDrive/PRISM'

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install librosa soundfile scipy scikit-learn rich loguru pyyaml pandas numpy

In [ ]:
import os

os.chdir('/content')

# Copy code from Drive
!cp -r {DRIVE_ROOT}/prism-colab/* /content/

# Verify model code is present
!ls models/cough_detector/
!ls models/shared/

## 2. Extract Features Dataset

In [ ]:
import zipfile

os.makedirs('../datasets/features', exist_ok=True)

zip_path = f'{DRIVE_ROOT}/datasets-features.zip'
print(f'Extracting {zip_path}...')

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall('../datasets/features')

# Verify
!ls datasets/features/ | head -5
!wc -l datasets/features/manifest.csv

In [ ]:
# Copy pre-trained checkpoint
os.makedirs('../models/checkpoints', exist_ok=True)
!cp {DRIVE_ROOT}/models/checkpoints/cough_detector_best.pt models/checkpoints/
!ls -la models/checkpoints/

## 3. Verify GPU & Test Augmentations

In [ ]:
import torch

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# Quick test: augmentations work
import numpy as np

from models.shared.waveform_augment import MicAugment

aug = MicAugment(p=1.0)
dummy = np.random.randn(48000).astype(np.float32) * 0.5
result = aug(dummy)
print(f'Augmentation test: shape={result.shape}, range=[{result.min():.3f}, {result.max():.3f}]')
print('OK!')

## 4. Dry Run (1-batch sanity check)

In [ ]:
from models.cough_detector.finetune import dry_run
from models.cough_detector.finetune_dataset import create_finetune_loaders
from models.cough_detector.model import build_model
from models.shared.checkpoint import load_checkpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Create a small loader for dry run
loaders = create_finetune_loaders(
    batch_size=4,
    num_workers=2,
    augment_prob=1.0,
)

model = build_model(pretrained=False, device=device)
load_checkpoint('../models/checkpoints/cough_detector_best.pt', model, device=device)

dry_run(model, loaders['train'], device)

## 5. Fine-Tune Training

This will:
- **Epochs 1-5**: Freeze conv1, bn1, layer1, layer2 (train only later layers + heads)
- **Epochs 6-15**: Unfreeze everything, LR reduced by 10×
- Save best checkpoint to `cough_detector_finetuned.pt`

In [ ]:
from models.cough_detector.finetune import FineTuneTrainer
from models.cough_detector.finetune_dataset import create_finetune_loaders
from models.cough_detector.model import build_model
from models.shared.checkpoint import load_checkpoint

device = torch.device('cuda')
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Create data loaders with mic augmentation
print('Creating fine-tune data loaders...')
loaders = create_finetune_loaders(
    batch_size=64,
    num_workers=4,
    augment_prob=0.8,
)

for split, loader in loaders.items():
    print(f'  {split}: {len(loader.dataset)} samples')

# Load pre-trained model
print('\nLoading pre-trained model...')
model = build_model(pretrained=False, device=device)
info = load_checkpoint('../models/checkpoints/cough_detector_best.pt', model, device=device)
print(f'  Loaded epoch {info["epoch"]}, AUC={info["metrics"].get("auc", "N/A")}')

# Fine-tune
trainer = FineTuneTrainer(
    model=model,
    train_loader=loaders['train'],
    val_loader=loaders['val'],
    device=device,
    lr=1e-4,
    epochs=15,
    freeze_epochs=5,
    patience=5,
)

best_metrics = trainer.train()
print(f'\nBest metrics: {best_metrics}')

## 6. Evaluate on Original Test Set

Verify no catastrophic forgetting — AUC should stay ≥ 0.85 on clean audio.

In [ ]:
from models.cough_detector.dataset import create_dataloaders
from models.cough_detector.evaluate import evaluate
from models.shared.checkpoint import load_checkpoint

# Load finetuned model
model_ft = build_model(pretrained=False, device=device)
load_checkpoint('../models/checkpoints/cough_detector_finetuned.pt', model_ft, device=device)

# Evaluate on ORIGINAL (non-augmented) test set
original_loaders = create_dataloaders(
    batch_size=64,
    num_workers=4,
    weighted_sampling=False,
)

print('Evaluating on original (clean) test set...')
clean_metrics = evaluate(model_ft, original_loaders['test'], device)
print(f'Clean test AUC: {clean_metrics["auc"]:.4f}')
print(f'Clean test F1:  {clean_metrics["f1"]:.4f}')
print(f'Confusion matrix: {clean_metrics["confusion_matrix"]}')

if clean_metrics['auc'] >= 0.85:
    print('\n✅ No catastrophic forgetting — clean AUC is still good!')
else:
    print('\n⚠️ Warning: Clean AUC dropped below 0.85 — may need to reduce augmentation intensity.')

## 7. Evaluate on Mic-Augmented Test Set

Test how well the model handles mic-like audio.

In [ ]:
# Evaluate on mic-augmented test set
ft_loaders = create_finetune_loaders(
    batch_size=64,
    num_workers=4,
    augment_prob=1.0,  # 100% augmented for testing
)

print('Evaluating on mic-augmented test set...')
mic_metrics = evaluate(model_ft, ft_loaders['test'], device)
print(f'Mic-augmented test AUC: {mic_metrics["auc"]:.4f}')
print(f'Mic-augmented test F1:  {mic_metrics["f1"]:.4f}')

# Compare with original model on mic-augmented data
print('\n--- Comparison: Original vs Fine-tuned on mic audio ---')
model_orig = build_model(pretrained=False, device=device)
load_checkpoint('../models/checkpoints/cough_detector_best.pt', model_orig, device=device)

orig_mic_metrics = evaluate(model_orig, ft_loaders['test'], device)
print(f'Original model on mic audio:   AUC={orig_mic_metrics["auc"]:.4f}')
print(f'Fine-tuned model on mic audio:  AUC={mic_metrics["auc"]:.4f}')
print(f'Improvement: {mic_metrics["auc"] - orig_mic_metrics["auc"]:+.4f}')

## 8. Save Results & Download

In [ ]:
import json

# Save eval results
results = {
    'fine_tune_best_val': best_metrics,
    'clean_test': {k: v for k, v in clean_metrics.items() if k != 'confusion_matrix'},
    'mic_test': {k: v for k, v in mic_metrics.items() if k != 'confusion_matrix'},
    'original_on_mic': {k: v for k, v in orig_mic_metrics.items() if k != 'confusion_matrix'},
}

with open('../models/checkpoints/finetune_eval.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

In [ ]:
# Copy checkpoint and eval to Google Drive
!cp models/checkpoints/cough_detector_finetuned.pt {DRIVE_ROOT}/models/checkpoints/
!cp models/checkpoints/finetune_eval.json {DRIVE_ROOT}/models/checkpoints/

print('\n✅ Checkpoint and evaluation saved to Google Drive!')
print(f'   {DRIVE_ROOT}/models/checkpoints/cough_detector_finetuned.pt')
print(f'   {DRIVE_ROOT}/models/checkpoints/finetune_eval.json')
print('\nCopy these to your local PRISM project:')
print('   models/checkpoints/cough_detector_finetuned.pt')
print('   models/checkpoints/finetune_eval.json')

In [ ]:
# Optional: download directly
from google.colab import files

files.download('../models/checkpoints/cough_detector_finetuned.pt')
files.download('../models/checkpoints/finetune_eval.json')